In [1]:
!pip install yfinance pandas numpy


In [4]:
import yfinance as yf
import pandas as pd
import numpy as np


In [5]:
ticker = yf.Ticker("AAPL")

income_stmt = ticker.financials
balance_sheet = ticker.balance_sheet
cash_flow = ticker.cashflow

income_stmt.head()


,2025-09-30,2024-09-30,2023-09-30,2022-09-30,2021-09-30
Tax Effect Of Unusual Items,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
Tax Rate For Calcs,1.560000e-01,2.410000e-01,1.470000e-01,1.620000e-01,NaN
Normalized EBITDA,1.447480e+11,1.346610e+11,1.258200e+11,1.305410e+11,NaN
Net Income From Continuing Operation Net Minority Interest,1.120100e+11,9.373600e+10,9.699500e+10,9.980300e+10,NaN
Reconciled Depreciation,1.169800e+10,1.144500e+10,1.151900e+10,1.110400e+10,NaN


In [6]:
import yfinance as yf
import pandas as pd
import numpy as np

# Select ticker
ticker_symbol = "AAPL"
ticker = yf.Ticker(ticker_symbol)

# Fetch statements
income_stmt = ticker.financials
balance_sheet = ticker.balance_sheet
cash_flow = ticker.cashflow

# Get latest available year for each statement
latest_income_year = income_stmt.columns[0]
latest_balance_year = balance_sheet.columns[0]
latest_cash_year = cash_flow.columns[0]

# -------- Extract Metrics Safely -------- #

# Revenue
revenue = income_stmt.loc["Total Revenue", latest_income_year] \
    if "Total Revenue" in income_stmt.index else None

# Net Income (handle possible naming differences)
if "Net Income" in income_stmt.index:
    net_income = income_stmt.loc["Net Income", latest_income_year]
elif "Net Income From Continuing Operations" in income_stmt.index:
    net_income = income_stmt.loc["Net Income From Continuing Operations", latest_income_year]
else:
    net_income = None

# Total Debt
total_debt = balance_sheet.loc["Total Debt", latest_balance_year] \
    if "Total Debt" in balance_sheet.index else None

# Operating Cash Flow
if "Operating Cash Flow" in cash_flow.index:
    operating_cash_flow = cash_flow.loc["Operating Cash Flow", latest_cash_year]
elif "Cash Flow From Continuing Operating Activities" in cash_flow.index:
    operating_cash_flow = cash_flow.loc["Cash Flow From Continuing Operating Activities", latest_cash_year]
else:
    operating_cash_flow = None

# -------- Print Raw Values -------- #

print("Ticker:", ticker_symbol)
print("Revenue:", revenue)
print("Net Income:", net_income)
print("Total Debt:", total_debt)
print("Operating Cash Flow:", operating_cash_flow)


Ticker: AAPL
Revenue: 416161000000.0
Net Income: 112010000000.0
Total Debt: 98657000000.0
Operating Cash Flow: 111482000000.0


In [7]:
# Function to convert large numbers to Billions format
def to_billions(value):
    if value is None:
        return None
    return f"${value / 1e9:.2f}B"

# Convert values
revenue_b = to_billions(revenue)
net_income_b = to_billions(net_income)
debt_b = to_billions(total_debt)
cashflow_b = to_billions(operating_cash_flow)

print("Formatted Values:")
print("Revenue:", revenue_b)
print("Net Income:", net_income_b)
print("Total Debt:", debt_b)
print("Operating Cash Flow:", cashflow_b)


Formatted Values:
Revenue: $416.16B
Net Income: $112.01B
Total Debt: $98.66B
Operating Cash Flow: $111.48B


In [8]:
# Get two most recent revenue values
revenues = income_stmt.loc["Total Revenue"]

# Ensure at least 2 years exist
if len(revenues) >= 2:
    current_rev = revenues.iloc[0]
    previous_rev = revenues.iloc[1]

    yoy_growth = ((current_rev - previous_rev) / previous_rev) * 100
    yoy_growth = round(yoy_growth, 2)
else:
    yoy_growth = None

print("YoY Growth:", yoy_growth, "%")


YoY Growth: 6.43 %


In [9]:
# Determine cash flow status
if operating_cash_flow is not None:
    cash_flow_status = "Positive" if operating_cash_flow > 0 else "Negative"
else:
    cash_flow_status = "Unknown"

print("Cash Flow Status:", cash_flow_status)


Cash Flow Status: Positive


In [10]:
structured_input = f"""
Revenue: {revenue_b}
Net Income: {net_income_b}
Debt: {debt_b}
YoY Growth: {yoy_growth}%
Operating Cash Flow: {cash_flow_status}
"""

print(structured_input)



Revenue: $416.16B
Net Income: $112.01B
Debt: $98.66B
YoY Growth: 6.43%
Operating Cash Flow: Positive



In [11]:
# Convert formatted strings back to numeric for scoring
revenue_num = revenue
net_income_num = net_income
debt_num = total_debt
cashflow_num = operating_cash_flow

# ---------------- Risk Scoring Logic ---------------- #

risk_score = 50  # Start neutral

# Growth impact
if yoy_growth is not None:
    if yoy_growth > 15:
        risk_score -= 10
    elif yoy_growth < 0:
        risk_score += 15

# Profitability impact
if net_income_num is not None:
    if net_income_num > 0:
        risk_score -= 10
    else:
        risk_score += 20

# Debt impact (compare to revenue)
if revenue_num and debt_num:
    debt_ratio = debt_num / revenue_num
    if debt_ratio > 0.6:
        risk_score += 15
    elif debt_ratio < 0.3:
        risk_score -= 5

# Cash flow impact
if cashflow_num:
    if cashflow_num > 0:
        risk_score -= 5
    else:
        risk_score += 10

# Clamp score between 0 and 100
risk_score = max(0, min(100, risk_score))

# Investment recommendation
if risk_score <= 40:
    recommendation = "Buy"
elif risk_score <= 65:
    recommendation = "Hold"
else:
    recommendation = "Risky"

# ---------------- Generate Report ---------------- #

report = f"""
Financial Analysis Report

1. Revenue & Growth Analysis:
The company reported revenue of {revenue_b} with a year-over-year growth of {yoy_growth}%, indicating {'strong expansion' if yoy_growth > 10 else 'moderate growth'}.

2. Profitability Assessment:
Net income stands at {net_income_b}, reflecting {'solid profitability' if net_income_num > 0 else 'operational losses'}.

3. Leverage & Risk Analysis:
Total debt amounts to {debt_b}, suggesting {'manageable leverage' if debt_ratio < 0.5 else 'elevated financial leverage'}.

4. Liquidity Position:
Operating cash flow is {cashflow_b}, indicating {'healthy liquidity' if cashflow_num > 0 else 'liquidity concerns'}.

5. Overall Risk Score: {risk_score}/100

6. Investment Recommendation: {recommendation}

7. Final Outlook:
Based on current financial metrics, the company presents a {recommendation.lower()} investment profile with balanced growth and risk factors.
"""

print(report)



Financial Analysis Report

1. Revenue & Growth Analysis:
The company reported revenue of $416.16B with a year-over-year growth of 6.43%, indicating moderate growth.

2. Profitability Assessment:
Net income stands at $112.01B, reflecting solid profitability.

3. Leverage & Risk Analysis:
Total debt amounts to $98.66B, suggesting manageable leverage.

4. Liquidity Position:
Operating cash flow is $111.48B, indicating healthy liquidity.

5. Overall Risk Score: 30/100

6. Investment Recommendation: Buy

7. Final Outlook:
Based on current financial metrics, the company presents a buy investment profile with balanced growth and risk factors.



In [12]:
import yfinance as yf
import pandas as pd
import json
import time

# ---------------- Helper Functions ---------------- #

def to_billions(value):
    if value is None:
        return None
    return f"${value / 1e9:.2f}B"

def calculate_yoy(income_stmt):
    if "Total Revenue" not in income_stmt.index:
        return None
    revenues = income_stmt.loc["Total Revenue"]
    if len(revenues) < 2:
        return None
    current = revenues.iloc[0]
    previous = revenues.iloc[1]
    return round(((current - previous) / previous) * 100, 2)

# ---------------- Risk + Report Generator ---------------- #

def generate_report(revenue, net_income, debt, cashflow, yoy):
    risk_score = 50

    if yoy is not None:
        if yoy > 15:
            risk_score -= 10
        elif yoy < 0:
            risk_score += 15

    if net_income > 0:
        risk_score -= 10
    else:
        risk_score += 20

    if revenue and debt:
        debt_ratio = debt / revenue
        if debt_ratio > 0.6:
            risk_score += 15
        elif debt_ratio < 0.3:
            risk_score -= 5
    else:
        debt_ratio = 0

    if cashflow > 0:
        risk_score -= 5
    else:
        risk_score += 10

    risk_score = max(0, min(100, risk_score))

    if risk_score <= 40:
        recommendation = "Buy"
    elif risk_score <= 65:
        recommendation = "Hold"
    else:
        recommendation = "Risky"

    report = f"""
Financial Analysis Report

1. Revenue & Growth Analysis:
The company reported revenue of {to_billions(revenue)} with a year-over-year growth of {yoy}%, indicating {'strong expansion' if yoy and yoy > 10 else 'moderate growth'}.

2. Profitability Assessment:
Net income stands at {to_billions(net_income)}, reflecting {'solid profitability' if net_income > 0 else 'operational losses'}.

3. Leverage & Risk Analysis:
Total debt amounts to {to_billions(debt)}, suggesting {'manageable leverage' if debt_ratio < 0.5 else 'elevated financial leverage'}.

4. Liquidity Position:
Operating cash flow is {to_billions(cashflow)}, indicating {'healthy liquidity' if cashflow > 0 else 'liquidity concerns'}.

5. Overall Risk Score: {risk_score}/100

6. Investment Recommendation: {recommendation}

7. Final Outlook:
Based on current financial metrics, the company presents a {recommendation.lower()} investment profile with balanced growth and risk factors.
"""
    return report

# ---------------- Ticker List ---------------- #

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA",
           "META", "NVDA", "JPM", "BAC", "WMT",
           "V", "MA", "UNH", "HD", "PG",
           "KO", "PEP", "XOM", "CVX", "INTC"]

dataset = []

# ---------------- Main Loop ---------------- #

for symbol in tickers:
    try:
        ticker = yf.Ticker(symbol)
        income = ticker.financials
        balance = ticker.balance_sheet
        cash = ticker.cashflow

        latest_income = income.columns[0]
        latest_balance = balance.columns[0]
        latest_cash = cash.columns[0]

        revenue = income.loc["Total Revenue", latest_income]
        net_income = income.loc["Net Income", latest_income]
        debt = balance.loc["Total Debt", latest_balance]
        cashflow = cash.loc["Operating Cash Flow", latest_cash]
        yoy = calculate_yoy(income)

        cash_status = "Positive" if cashflow > 0 else "Negative"

        structured_input = f"""
Revenue: {to_billions(revenue)}
Net Income: {to_billions(net_income)}
Debt: {to_billions(debt)}
YoY Growth: {yoy}%
Operating Cash Flow: {cash_status}
"""

        report = generate_report(revenue, net_income, debt, cashflow, yoy)

        dataset.append({
            "instruction": "Generate a financial analysis report",
            "input": structured_input.strip(),
            "output": report.strip()
        })

        print(f"Processed {symbol}")
        time.sleep(1)

    except Exception as e:
        print(f"Skipping {symbol} due to error: {e}")

# Save dataset
with open("finreport_dataset.jsonl", "w") as f:
    for entry in dataset:
        f.write(json.dumps(entry) + "\n")

print("Dataset creation complete.")
print("Total samples:", len(dataset))


Processed AAPL
Processed MSFT
Processed GOOGL
Processed AMZN
Processed TSLA
Processed META
Processed NVDA
Processed JPM
Processed BAC
Processed WMT
Processed V
Processed MA
Processed UNH
Processed HD
Processed PG
Processed KO
Processed PEP
Processed XOM
Processed CVX
Processed INTC
Dataset creation complete.
Total samples: 20


In [13]:
# Alternative S&P 500 CSV source
url = "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv"

sp500_table = pd.read_csv(url)

sp500_tickers = sp500_table["Symbol"].tolist()

print("Total tickers fetched:", len(sp500_tickers))
print(sp500_tickers[:10])


Total tickers fetched: 503
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [14]:
# Fix ticker format for Yahoo Finance
cleaned_tickers = []

for t in sp500_tickers:
    if "." in t:
        t = t.replace(".", "-")
    cleaned_tickers.append(t)

print("Sample cleaned tickers:", cleaned_tickers[:20])


Sample cleaned tickers: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL']


In [1]:
import yfinance as yf
import pandas as pd
import json
import time

# ---------------- Step 1: Get S&P 500 Tickers ---------------- #

url = "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv"
sp500_table = pd.read_csv(url)
sp500_tickers = sp500_table["Symbol"].tolist()

# Fix dot tickers (e.g., BRK.B -> BRK-B)
cleaned_tickers = []
for t in sp500_tickers:
    if "." in t:
        t = t.replace(".", "-")
    cleaned_tickers.append(t)

# Limit to first 200 companies for stability
tickers = cleaned_tickers[:200]

print("Total tickers selected:", len(tickers))

# ---------------- Helper Functions ---------------- #

def to_billions(value):
    if value is None:
        return None
    return f"${value / 1e9:.2f}B"

def calculate_yoy(income_stmt):
    if "Total Revenue" not in income_stmt.index:
        return None
    revenues = income_stmt.loc["Total Revenue"]
    if len(revenues) < 2:
        return None
    current = revenues.iloc[0]
    previous = revenues.iloc[1]
    return round(((current - previous) / previous) * 100, 2)

def generate_report(revenue, net_income, debt, cashflow, yoy):
    risk_score = 50

    # Growth impact
    if yoy is not None:
        if yoy > 15:
            risk_score -= 10
        elif yoy < 0:
            risk_score += 15

    # Profitability impact
    if net_income > 0:
        risk_score -= 10
    else:
        risk_score += 20

    # Debt impact
    if revenue and debt:
        debt_ratio = debt / revenue
        if debt_ratio > 0.6:
            risk_score += 15
        elif debt_ratio < 0.3:
            risk_score -= 5
    else:
        debt_ratio = 0

    # Cash flow impact
    if cashflow > 0:
        risk_score -= 5
    else:
        risk_score += 10

    risk_score = max(0, min(100, risk_score))

    if risk_score <= 40:
        recommendation = "Buy"
    elif risk_score <= 65:
        recommendation = "Hold"
    else:
        recommendation = "Risky"

    report = f"""
Financial Analysis Report

1. Revenue & Growth Analysis:
The company reported revenue of {to_billions(revenue)} with a year-over-year growth of {yoy}%, indicating {'strong expansion' if yoy and yoy > 10 else 'moderate growth'}.

2. Profitability Assessment:
Net income stands at {to_billions(net_income)}, reflecting {'solid profitability' if net_income > 0 else 'operational losses'}.

3. Leverage & Risk Analysis:
Total debt amounts to {to_billions(debt)}, suggesting {'manageable leverage' if debt_ratio < 0.5 else 'elevated financial leverage'}.

4. Liquidity Position:
Operating cash flow is {to_billions(cashflow)}, indicating {'healthy liquidity' if cashflow > 0 else 'liquidity concerns'}.

5. Overall Risk Score: {risk_score}/100

6. Investment Recommendation: {recommendation}

7. Final Outlook:
Based on current financial metrics, the company presents a {recommendation.lower()} investment profile with balanced growth and risk factors.
"""
    return report

# ---------------- Main Dataset Loop ---------------- #

dataset = []

for symbol in tickers:
    try:
        ticker = yf.Ticker(symbol)
        income = ticker.financials
        balance = ticker.balance_sheet
        cash = ticker.cashflow

        if income.empty or balance.empty or cash.empty:
            continue

        latest_income = income.columns[0]
        latest_balance = balance.columns[0]
        latest_cash = cash.columns[0]

        if "Total Revenue" not in income.index:
            continue
        if "Net Income" not in income.index:
            continue
        if "Total Debt" not in balance.index:
            continue
        if "Operating Cash Flow" not in cash.index:
            continue

        revenue = income.loc["Total Revenue", latest_income]
        net_income = income.loc["Net Income", latest_income]
        debt = balance.loc["Total Debt", latest_balance]
        cashflow = cash.loc["Operating Cash Flow", latest_cash]
        yoy = calculate_yoy(income)

        if None in [revenue, net_income, debt, cashflow, yoy]:
            continue

        cash_status = "Positive" if cashflow > 0 else "Negative"

        structured_input = f"""
Revenue: {to_billions(revenue)}
Net Income: {to_billions(net_income)}
Debt: {to_billions(debt)}
YoY Growth: {yoy}%
Operating Cash Flow: {cash_status}
"""

        report = generate_report(revenue, net_income, debt, cashflow, yoy)

        dataset.append({
            "instruction": "Generate a financial analysis report",
            "input": structured_input.strip(),
            "output": report.strip()
        })

        print(f"Processed {symbol}")
        time.sleep(1)

    except Exception as e:
        print(f"Skipped {symbol}: {e}")

# ---------------- Save Dataset ---------------- #

with open("finreport_dataset.jsonl", "w") as f:
    for entry in dataset:
        f.write(json.dumps(entry) + "\n")

print("\nDataset creation complete.")
print("Total samples generated:", len(dataset))


Total tickers selected: 200
Processed MMM
Processed AOS
Processed ABT
Processed ABBV
Processed ACN
Processed ADBE
Processed AMD
Processed AES
Processed AFL
Processed A
Processed APD
Processed ABNB
Processed AKAM
Processed ALB
Processed ARE
Processed ALGN
Processed ALLE
Processed LNT
Processed ALL
Processed GOOGL
Processed GOOG
Processed MO
Processed AMZN
Processed AMCR
Processed AEE
Processed AEP
Processed AXP
Processed AIG
Processed AMT
Processed AWK
Processed AMP
Processed AME
Processed AMGN
Processed APH
Processed ADI
Processed AON
Processed APA
Processed APO
Processed AAPL
Processed AMAT
Processed APTV
Processed ACGL
Processed ADM
Processed ANET
Processed AJG
Processed AIZ
Processed T
Processed ATO
Processed ADSK
Processed ADP
Processed AZO
Processed AVB
Processed AVY
Processed AXON
Processed BKR
Processed BALL
Processed BAC
Processed BAX
Processed BDX
Processed BRK-B
Processed BBY
Processed TECH
Processed BIIB
Processed BLK
Processed BX
Processed XYZ
Processed BK
Processed BA
Proc

In [2]:
len(dataset)


199

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import json

with open("/content/drive/MyDrive/finreport_dataset.jsonl", "w") as f:
    for row in dataset:
        f.write(json.dumps(row) + "\n")

print("Saved permanently to Drive ✅")


Saved permanently to Drive ✅


In [5]:
!ls /content/drive/MyDrive/


 1742441600272.jpg
'Bio Exp Pics'
 Biology_Project.pdf
'Bio project'
 Classroom
'CLASS XII MS.png'
'Colab Notebooks'
 datasad
'Digital documentation notes-revised.gdoc'
'Document (6).gdoc'
 finreport_dataset.jsonl
'How do organisms reproduce part 2.gslides'
'imagerecognition-180329053211 (1).pdf'
 Images
 kabir.gdoc
'layered motion-54-61.pdf'
'MATHEMATICS ASSIGNMENT_class10_AP.gdoc'
'MATHS REVISION 1 PAPER.gdoc'
'Module 1.pptx'
 NEU-DET
'object detection (1).pdf'
 Optical_Flow_Full_Presentation.pptx
'Pre Board Exam class 10 2020-21.gdoc'
 processed-D9CD121F-821C-405C-A433-06DE3715C1B7.jpeg
 scratches_100.jpg
 scratches_101.jpg
 scratches_102.jpg
 scratches_103.jpg
 scratches_104.jpg
 scratches_105.jpg
 scratches_106.jpg
 scratches_107.jpg
 scratches_108.jpg
 scratches_109.jpg
 scratches_10.jpg
 scratches_110.jpg
 scratches_111.jpg
 scratches_112.jpg
 scratches_113.jpg
 scratches_114.jpg
 scratches_115.jpg
 scratches_116.jpg
 scratches_117.jpg
 scratches_118.jpg
 scratches_119.jpg
 scra